In [0]:
from pyspark.sql.functions import *
import os
import pandas as pd

In [0]:
path = "/Workspace/Users/kevyn.zap@hotmail.com/mba-novo-caged-data-engineering-free-edition/landing/CAGEDMOV202001.txt"

competencia = 202001

In [0]:
# Lê o arquivo .txt com delimitador ";" e usando a primeira linha como cabeçalho
df = (
    spark
    .read
    .format("csv")
    .option("delimiter", ";")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("inferSchema", False)
    .load(path))

display(df)

In [0]:
df_bronze = (
    df
    .withColumn("_competencia_referencia", lit("202001"))
    .withColumn("_arquivo_origem", lit("CAGEDMOV202001.7z"))
    .withColumn("_caminho_origem", lit(path))
    .withColumn("_data_ingestao", current_timestamp())
)

print(f"Total de colunas: {len(df_bronze.columns)}")

df_bronze.select(
    "competênciamov",
    "uf",
    "cbo2002ocupação",
    "saldomovimentação",
    "_competencia_referencia",
    "_arquivo_origem",
    "_data_ingestao"
).show(5, truncate=False)

In [0]:
# Quantidade total de registros
total_registros = df_bronze.count()

# Quantidade de registros com competência divergente ou nula
registros_invalidos = df_bronze.filter(
    col("competênciamov").isNull()
    | (col("competênciamov") != competencia)
    | col("_competencia_referencia").isNull()
    | (col("_competencia_referencia") != competencia)
).count()

# Validações obrigatórias
assert total_registros > 0, (
    "O arquivo não contém registros."
)

assert registros_invalidos == 0, (
    f"Foram encontrados {registros_invalidos} registros "
    f"com competência inválida para {competencia}."
)

print("Validação concluída com sucesso!")
print(f"Competência: {competencia}")
print(f"Total de registros: {total_registros:,}")
print(f"Registros inválidos: {registros_invalidos:,}")

In [0]:
# Tabela Bronze registrada no Unity Catalog
tabela_bronze = (
    "workspace.bronze.novo_caged_movimentacoes"
)

# Proteção contra gravação de competência incorreta
assert total_registros > 0, "Não existem registros para gravar."
assert registros_invalidos == 0, "Existem registros inválidos."

# Sobrescrever somente a competência selecionada
# Na primeira execução a tabela ainda não existe: criar antes de sobrescrever
tabela_existe = (
    spark.sql("SHOW TABLES IN workspace.bronze LIKE 'novo_caged_movimentacoes'")
    .count() > 0
)

if tabela_existe:
    (
        df_bronze.writeTo(tabela_bronze)
        .using("delta")
        .overwrite(
            col("_competencia_referencia") == competencia
        )
    )
else:
    (
        df_bronze.writeTo(tabela_bronze)
        .using("delta")
        .create()
    )

print(f"Competência {competencia} reprocessada com sucesso!")

In [0]:
# Validar a tabela após a gravação
df_validacao = spark.table(tabela_bronze)

total_bronze = df_validacao.count()

total_competencia = df_validacao.filter(
    col("_competencia_referencia") == competencia
).count()

# Conferir se a competência gravada está completa
assert total_competencia == total_registros, (
    "A quantidade gravada difere da quantidade de origem."
)

print("Validação da Bronze concluída!")
print(f"Competência: {competencia}")
print(f"Registros da competência: {total_competencia:,}")
print(f"Total da tabela Bronze: {total_bronze:,}")
print(f"Quantidade de colunas: {len(df_validacao.columns)}")

# Consultar o histórico de transações Delta
display(
    spark.sql(f"DESCRIBE HISTORY {tabela_bronze}")
    .select("version", "timestamp", "operation", "operationParameters")
    .limit(5)
)